# 2-2. 날씨 x 소비 시계열 분석

경기도 카드소비 x 날씨 데이터 EDA 2단계 (2차).
목표:
1. 권역별(남부/동부/북부) 실제 기온·날씨 차이가 있는지 확인
2. 기온 구간별 소비 비교 (분류등급별)
3. 날씨 이벤트(강수/폭염/한파) vs 평상시 비교 (분류등급별, t-test 포함)
4. 일별 매출 추이 x 기온 이중 트렌드 시각화

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats

system = platform.system()
if system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False

pd.set_option("display.max_columns", None)
%matplotlib inline

## 2-2-0. 데이터 로드

In [ ]:
from pathlib import Path

# src/analysis/ 에서 실행하는 걸 기준으로 프로젝트 루트까지 2단계 상위 경로.
CATEGORY_DIR = Path("../../data/processed/consume_weather_by_category")

GROUP_ORDER = ["필수", "애매", "불필요"]
REGION_ORDER = ["남부", "동부", "북부"]

group_dfs = {}
for group in GROUP_ORDER:
    g_df = pd.read_parquet(CATEGORY_DIR / f"{group}.parquet")
    g_df["분류등급"] = group
    group_dfs[group] = g_df

df = pd.concat(group_dfs.values(), ignore_index=True)
df["date"] = pd.to_datetime(df["date"])

print("shape:", df.shape)
print("권역:", sorted(df["지점명"].unique()))
df.head()

## 2-2-1. 권역별 실제 날씨 차이 확인

남부/동부/북부가 실제로 다른 날씨를 보이는지 먼저 확인한다.
(같은 날짜엔 같은 권역이면 같은 날씨값이 반복되므로, 날짜+지점명 기준 중복 제거 후 비교)

In [ ]:
weather_cols = ["평균기온(°C)", "일강수량(mm)", "평균 풍속(m/s)", "일 최심적설(cm)"]
weather_unique = df.drop_duplicates(subset=["date", "지점명"])[["지점명"] + weather_cols]

print(weather_unique.groupby("지점명")[weather_cols].describe().T)

In [ ]:
# 시각화 도화지(Canvas) 준비 및 권역별 날씨 비교

# 4개의 그래프 축(axes)과 4개의 날씨 컬럼(weather_cols)을 하나씩 짝지어(zip) 순회
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, weather_cols):
    box_data = [weather_unique.loc[weather_unique["지점명"] == r, col] for r in REGION_ORDER]
    ax.boxplot(box_data, tick_labels=REGION_ORDER, showfliers=False)
    ax.set_title(col)
    ax.set_ylabel(col)

plt.suptitle("권역별 날씨 변수 비교 (이상치 제외 boxplot)")
plt.tight_layout()
plt.show()

# 통계적 검정: 권역별 평균기온 일원분산분석 (ANOVA)

# 권역간 평균기온 차이가 통계적으로 유의한지 ANOVA로 확인
groups = [weather_unique.loc[weather_unique["지점명"] == r, "평균기온(°C)"] for r in REGION_ORDER]
# scipy.stats의 f_oneway 함수를 사용하여 일원분산분석(One-way ANOVA)을 수행
f_stat, p_value = stats.f_oneway(*groups)

print(f"평균기온 권역간 ANOVA: F={f_stat:.2f}, p-value={p_value:.4f}")
print("-> 유의수준 0.05 기준", "유의미한 차이 있음" if p_value < 0.05 else "유의미한 차이 없음")

## 2-2-2. 기온 구간별 소비 비교 (분류등급별)

기온을 구간화해서, 구간별 평균 소비(로그변환)가 분류등급별로 어떻게 다른지 비교한다.

In [ ]:
# 기온 구간(Binning) 정의 및 범주형 변수 생성

temp_bins = [-100, 0, 10, 20, 30, 100]
temp_labels = ["영하", "0~10도", "10~20도", "20~30도", "30도 이상"]

# [데이터 이산화] pd.cut 함수를 사용하여 '평균기온(°C)' 컬럼의 
# 수치들을 정의한 경계값에 맞춰 '기온구간'이라는 범주형(Categorical) 파생 변수로 변환
df["기온구간"] = pd.cut(df["평균기온(°C)"], bins=temp_bins, labels=temp_labels)

# 기온구간별 / 분류등급별 매출 통계량 집계
temp_stats = (
    df.groupby(["기온구간", "분류등급"], observed=True)["매출금액"]
    .apply(lambda x: np.log1p(x).mean())
    .reset_index()
    .pivot(index="기온구간", columns="분류등급", values="매출금액")
    .reindex(temp_labels)
)

# 시각화 도화지 준비 및 다중 막대그래프 생성
fig, ax = plt.subplots(figsize=(9, 5))
temp_stats[GROUP_ORDER].plot(kind="bar", ax=ax)
ax.set_title("기온구간별 평균 log(매출금액+1) (분류등급별)")
ax.set_xlabel("기온구간")
ax.set_ylabel("평균 log(매출금액 + 1)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 필수 그룹 안에서 소분류별로 기온 반응이 다른지 확인 (상위 8개 소분류만)
essential = df[df["분류등급"] == "필수"]

# 필수 그룹 내에서 각 '소분류'별로 매출금액의 전체 총합을 구한 뒤, 
# 매출 규모가 가장 큰 상위 8개 품목을 골라 그 이름(인덱스)만 리스트 형태로 저장
top_categories = essential.groupby("소분류")["매출금액"].sum().nlargest(8).index

# 시각화 도화지 준비 및 소분류별 기온 반응 선 그래프 생성
fig, ax = plt.subplots(figsize=(11, 5))
for cat in top_categories:
    sub = essential[essential["소분류"] == cat]
    cat_stats = sub.groupby("기온구간", observed=True)["매출금액"].apply(lambda x: np.log1p(x).mean()).reindex(temp_labels)
    ax.plot(temp_labels, cat_stats, marker="o", label=cat)

# 그래프 스타일링 및 레이아웃 최적화
ax.set_title("필수 그룹 상위 소분류별 기온구간 반응")
ax.set_xlabel("기온구간")
ax.set_ylabel("평균 log(매출금액 + 1)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2-2-3. 날씨 이벤트 vs 평상시 비교 (분류등급별, t-test)

- 강수일: 일강수량 > 0
- 폭염일: 평균기온 >= 33도
- 한파일: 평균기온 <= -10도

각 이벤트일과 평상시(해당 이벤트가 아닌 날)의 log(매출금액) 평균을 분류등급별로 비교하고,
독립표본 t-test로 차이가 통계적으로 유의한지 확인한다.

In [ ]:
# 특수 기상 이벤트(날씨 조건) 기준 정의

# [강수일 정의]
df["is_rain"] = df["일강수량(mm)"] > 0
# [폭염일 정의]
df["is_heatwave"] = df["평균기온(°C)"] >= 33
# [한파일 정의]
df["is_coldwave"] = df["평균기온(°C)"] <= -10

events = {"강수일": "is_rain", "폭염일": "is_heatwave", "한파일": "is_coldwave"}

results = []

# 기상 이벤트별 / 분류등급별 집단 분리 및 t-검정
# 정의한 기상 이벤트 3개를 하나씩 순회
for event_name, col in events.items():
    for g in GROUP_ORDER:
        sub = df[df["분류등급"] == g]
        # [이벤트일 매출 로그 변환]
        event_vals = np.log1p(sub.loc[sub[col], "매출금액"])
        # [평상시 매출 로그 변환]
        normal_vals = np.log1p(sub.loc[~sub[col], "매출금액"])

        # [안전 장치] 두 집단 중 어느 한쪽이라도 표본 수가 2개 미만이면 
        # 통계 검정(t-test)이 불가능하므로, 계산을 건너뛰기
        if len(event_vals) < 2 or len(normal_vals) < 2:
            continue

        t_stat, p_value = stats.ttest_ind(event_vals, normal_vals, equal_var=False)
        results.append({
            "이벤트": event_name,
            "분류등급": g,
            "이벤트일_평균": event_vals.mean(),
            "평상시_평균": normal_vals.mean(),
            "차이": event_vals.mean() - normal_vals.mean(),
            "t_stat": t_stat,
            "p_value": p_value,
            "유의함(p<0.05)": p_value < 0.05,
        })

# 종합 결과 테이블(DataFrame) 생성 및 출력
result_df = pd.DataFrame(results)
result_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

# 기상 이벤트별 다중 막대그래프(Side-by-Side) 생성
# 3개의 그래프 축(axes)과 3개의 기상 이벤트 조건을 하나씩 짝지어(zip) 순회
for ax, (event_name, col) in zip(axes, events.items()):
    sub_result = result_df[result_df["이벤트"] == event_name]
    x = np.arange(len(GROUP_ORDER))
    width = 0.35
    # [이벤트일 막대]
    ax.bar(x - width/2, sub_result["이벤트일_평균"], width, label="이벤트일")
    # [평상시 막대]
    ax.bar(x + width/2, sub_result["평상시_평균"], width, label="평상시")
    ax.set_xticks(x)
    ax.set_xticklabels(GROUP_ORDER)
    ax.set_title(event_name)
    ax.legend(fontsize=8)

axes[0].set_ylabel("평균 log(매출금액 + 1)")
plt.tight_layout()
plt.show()

## 2-2-4. 일별 매출 추이 x 기온 이중 트렌드

필수 그룹의 일별 매출(7일 이동평균)과 평균기온을 같은 그래프에 겹쳐 그려서,
두 흐름이 같이 움직이는지 시각적으로 확인한다.

In [ ]:
# 전체 데이터에서 '분류등급'이 '필수'인 데이터만 필터링
essential = df[df["분류등급"] == "필수"]
# [필수 그룹 매출 집계]
daily_sales = essential.groupby("date")["매출금액"].sum().reset_index().sort_values("date")
# [전국 평균기온 집계]
daily_temp = df.drop_duplicates(subset=["date", "지점명"]).groupby("date")["평균기온(°C)"].mean().reset_index().sort_values("date")

fig, ax1 = plt.subplots(figsize=(14, 5))

# 첫 번째 Y축 : 필수 그룹 매출금액 추이
ax1.plot(daily_sales["date"], daily_sales["매출금액"].rolling(7, min_periods=1).mean(), color="steelblue", label="매출금액 (7일 이동평균)")
ax1.set_xlabel("날짜")
ax1.set_ylabel("매출금액 합계", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")

# 두 번째 Y축 : 이중 축(twinx)을 활용한 평균기온 추이
ax2 = ax1.twinx()
ax2.plot(daily_temp["date"], daily_temp["평균기온(°C)"], color="darkorange", alpha=0.7, label="평균기온")
ax2.set_ylabel("평균기온(°C)", color="darkorange")
ax2.tick_params(axis="y", labelcolor="darkorange")

# X축 날짜 포맷 및 레이아웃 최적화
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
plt.title("필수 그룹 일별 매출 추이 x 평균기온")
plt.tight_layout()
plt.show()